# GENIE Task 1 Image Baseline

This notebook is the mentor-facing Colab wrapper for the official image-based Task 1 baseline.
It uses `src/task1_autoencoder.py` as the source of truth and does not reimplement training logic inside the notebook.

Active detector channel order: `Tracks`, `ECAL`, `HCAL`.
Use `RUN_MODE = "sanity"` for a short 10-epoch check or `RUN_MODE = "full"` for the final long run.


In [ ]:
# 1. Optional Google Drive mount
from pathlib import Path

USE_GOOGLE_DRIVE = True
DRIVE_MOUNT_POINT = Path('/content/drive')

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT_POINT))
        print(f'Mounted Google Drive at {DRIVE_MOUNT_POINT}')
    except ImportError:
        print('google.colab is not available in this environment.')
else:
    print('Skipping Google Drive mount.')


In [ ]:
# 2. Clone repo and install dependencies
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nitik1998/GENIE_DiffusionLearning.git'
REPO_DIR = Path('/content/GENIE_DiffusionLearning')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Repo ready at: {REPO_DIR}')
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# 3. Imports and runtime config
from pathlib import Path
import argparse
import json
import os
import random
import shutil
import shlex
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, Markdown, display

from src.config import CHANNEL_NAMES, get_auto_batch_size
from src.data_utils import load_dataset, make_splits
from src.task1_autoencoder import (
    fit_task1_preprocessor,
    apply_task1_preprocessor,
    main as task1_main,
)

SEED = 42
FORCE_RERUN = True
RUN_MODE = "sanity"  # change to "full" for the final submission run
SANITY_EPOCHS = 10
FULL_EPOCHS = 200

BASE_TASK_CONFIG = {
    'base_exp_name': 'task1_image_baseline',
    'display_name': 'Task 1 Image Baseline',
    'batch_size': 0,
    'max_events': None,
    'preprocess_mode': 'detector_reference',
    'boost_channel': 0,
    'boost_factor': 1.5,
}

def detect_vram_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

def choose_batch_size(task_num, requested_batch_size):
    if requested_batch_size and requested_batch_size > 0:
        return int(requested_batch_size)
    return int(get_auto_batch_size(task_num))

TASK_CONFIG = dict(BASE_TASK_CONFIG)
TASK_CONFIG["epochs"] = SANITY_EPOCHS if RUN_MODE == "sanity" else FULL_EPOCHS
TASK_CONFIG['exp_name'] = f"{BASE_TASK_CONFIG['base_exp_name']}_sanity" if RUN_MODE == 'sanity' else BASE_TASK_CONFIG['base_exp_name']
TASK_CONFIG['batch_size'] = choose_batch_size(task_num=1, requested_batch_size=BASE_TASK_CONFIG['batch_size'])
TASK_CONFIG['vram_gb'] = round(detect_vram_gb(), 1)

DATASET_SOURCE = Path('/content/drive/MyDrive/quark-gluon_data-set_n139306.hdf5')
if not DATASET_SOURCE.exists():
    DATASET_SOURCE = Path('/content/quark-gluon_data-set_n139306.hdf5')

RESULTS_ROOT = Path('/content/results_colab')
CHECKPOINT_ROOT = RESULTS_ROOT / 'checkpoints'
DATA_DIR = REPO_DIR / 'data'
DATASET_TARGET = DATA_DIR / 'quark-gluon_data-set_n139306.hdf5'
FINAL_RESULTS_DIR = RESULTS_ROOT / 'ae' / TASK_CONFIG['exp_name']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR.mkdir(parents=True, exist_ok=True)
for path in [RESULTS_ROOT, CHECKPOINT_ROOT, FINAL_RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if DATASET_SOURCE.exists() and DATASET_SOURCE.resolve() != DATASET_TARGET.resolve():
    shutil.copy2(DATASET_SOURCE, DATASET_TARGET)
elif not DATASET_TARGET.exists():
    raise FileNotFoundError(f"Dataset not found at {DATASET_SOURCE} or {DATASET_TARGET}")

os.environ['GENIE_PROJECT_ROOT'] = str(REPO_DIR)
os.environ['GENIE_DATA_DIR'] = str(DATA_DIR)
os.environ['GENIE_OUTPUT_DIR'] = str(RESULTS_ROOT)
os.environ['GENIE_CHECKPOINT_DIR'] = str(CHECKPOINT_ROOT)

print(f"Python executable: {sys.executable}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM (GB): {TASK_CONFIG['vram_gb']}")
print(f"Run mode: {RUN_MODE}")
print(f"Active experiment: {TASK_CONFIG['exp_name']} ({TASK_CONFIG['display_name']})")
print(f"Selected batch size: {TASK_CONFIG['batch_size']}")
print('Training runs in-process below, so epoch logs should appear directly in this cell output.')
print('Estimate remaining time after the first epoch: epoch_time * remaining_epochs.')

def show_png(path, width=900):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing image: {path}')

def print_metrics(path, title):
    path = Path(path)
    if not path.exists():
        print(f'Missing metrics file: {path}')
        return
    payload = json.loads(path.read_text())
    metrics = payload.get('metrics', payload)
    display(Markdown(f'### {title}'))
    for key, value in metrics.items():
        print(f'{key}: {value}')


In [ ]:
# 4. Data sanity check
sanity_events = 256 if TASK_CONFIG['max_events'] is None else min(TASK_CONFIG['max_events'], 256)
X, y = load_dataset(str(DATA_DIR), max_events=sanity_events)
print('Loaded shape:', X.shape)
print('Labels shape:', y.shape)
print('Channel names:', CHANNEL_NAMES)

train_idx, _, _ = make_splits(X, y, seed=SEED)
preprocessor_params = fit_task1_preprocessor(
    X[train_idx],
    preprocess_mode=TASK_CONFIG['preprocess_mode'],
    boost_channel=TASK_CONFIG['boost_channel'],
    boost_factor=TASK_CONFIG['boost_factor'],
)
X_norm = apply_task1_preprocessor(X[:3], preprocessor_params)

fig, axes = plt.subplots(3, 6, figsize=(18, 9), squeeze=False)
for row in range(3):
    for ch in range(3):
        raw = X[row, ch]
        raw_vmax = max(np.percentile(raw[raw > 0], 99.5), 1e-6) if np.any(raw > 0) else 1e-6
        axes[row, ch].imshow(raw, cmap='hot', vmin=0, vmax=raw_vmax)
        axes[row, ch].axis('off')

        norm = X_norm[row, ch]
        norm_vmax = max(np.percentile(norm[norm > 0], 99.5), 1e-6) if np.any(norm > 0) else 1e-6
        axes[row, ch + 3].imshow(norm, cmap='hot', vmin=0, vmax=norm_vmax)
        axes[row, ch + 3].axis('off')

        if row == 0:
            axes[row, ch].set_title(f'Raw {CHANNEL_NAMES[ch]}')
            axes[row, ch + 3].set_title(f'Normalized {CHANNEL_NAMES[ch]}')

plt.tight_layout()
preview_path = FINAL_RESULTS_DIR / 'task1_input_sanity_check.png'
fig.savefig(preview_path, dpi=160, bbox_inches='tight')
plt.close(fig)

display(Image(filename=str(preview_path), width=1400))
print(f'Saved preview to: {preview_path}')


In [ ]:
# 5. Final Task 1 config
display(Markdown(f"## Active Config: `{TASK_CONFIG['exp_name']}`"))
for key, value in TASK_CONFIG.items():
    print(f'{key}: {value}')


In [ ]:
# 6. Run Task 1 through the official script with live Colab logs
cmd = [
    sys.executable,
    '-u',
    'src/task1_autoencoder.py',
    '--exp-name', TASK_CONFIG['exp_name'],
    '--epochs', str(TASK_CONFIG['epochs']),
    '--batch-size', str(TASK_CONFIG['batch_size']),
    '--seed', str(SEED),
    '--preprocess-mode', TASK_CONFIG['preprocess_mode'],
    '--boost-channel', str(TASK_CONFIG['boost_channel']),
    '--boost-factor', str(TASK_CONFIG['boost_factor']),
]
if FORCE_RERUN:
    cmd.append('--force-rerun')
if TASK_CONFIG.get('max_events') is not None:
    cmd += ['--max-events', str(TASK_CONFIG['max_events'])]

cmd_str = ' '.join(shlex.quote(part) for part in cmd)
print(f'Running: {cmd_str}')
print('Logs should stream live below. Use the first completed epoch to estimate remaining time.')
start_time = time.time()
exit_code = os.system(cmd_str)
elapsed = time.time() - start_time
if exit_code != 0:
    raise RuntimeError(f'Task 1 command failed with exit code {exit_code}')
print(f"Task 1 run finished in {elapsed / 60:.1f} minutes")


In [ ]:
# 7. Visualize final outputs inline
display(Markdown(f"## Final Outputs: `{TASK_CONFIG['exp_name']}`"))
show_png(FINAL_RESULTS_DIR / 'original_vs_reconstructed.png', width=1200)
show_png(FINAL_RESULTS_DIR / 'original_vs_reconstructed_vs_abs_error.png', width=1200)
show_png(FINAL_RESULTS_DIR / 'true_mask_vs_pred_mask.png', width=1200)
show_png(FINAL_RESULTS_DIR / 'sparse_diagnostics.png', width=1200)
show_png(FINAL_RESULTS_DIR / 'threshold_sweep.png', width=900)
show_png(FINAL_RESULTS_DIR / 'sparse_vs_full_metric_comparison.png', width=900)
show_png(FINAL_RESULTS_DIR / 'baseline_vs_final_comparison.png', width=1200)
show_png(FINAL_RESULTS_DIR / 'training_curves.png', width=900)
print_metrics(FINAL_RESULTS_DIR / 'metrics.json', 'Task 1 Metrics')


In [ ]:
# 8. Package artifacts for download
archive_base = RESULTS_ROOT / TASK_CONFIG['exp_name']
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=str(FINAL_RESULTS_DIR.parent), base_dir=FINAL_RESULTS_DIR.name)
print(f'Created archive: {archive_path}')

try:
    from google.colab import files
    files.download(archive_path)
except Exception as exc:
    print(f'Automatic download skipped: {exc}')
